In [1]:
from datetime import datetime

import numpy as np
import pandas as pd
import json

from preprocessing import year

/Users/lukas/dev/data_viz/assignement1.1/minard-visualization/src/data/preprocessing.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./irish-property-sales.csv', encoding="latin1")


In [2]:
df = pd.read_csv('./irish-property-sales.csv', encoding="latin1")

column_names = [
    'date_of_sale',
    'address',
    'county',
    'eircode',
    'price',
    'not_full_market_price',
    'vat_exclusive',
    'property_description',
    'propterty_size_description'
]


def get_county_province_mapping(county: str):
    county = county.lower().strip()
    county_to_province = {
        "galway": "Connacht",
        "leitrim": "Connacht",
        "mayo": "Connacht",
        "roscommon": "Connacht",
        "sligo": "Connacht",
        "carlow": "Leinster",
        "dublin": "Leinster",
        "kildare": "Leinster",
        "kilkenny": "Leinster",
        "laois": "Leinster",
        "longford": "Leinster",
        "louth": "Leinster",
        "meath": "Leinster",
        "offaly": "Leinster",
        "westmeath": "Leinster",
        "wexford": "Leinster",
        "wicklow": "Leinster",
        "clare": "Munster",
        "cork": "Munster",
        "kerry": "Munster",
        "limerick": "Munster",
        "tipperary": "Munster",
        "waterford": "Munster",
        "antrim": "Ulster",
        "armagh": "Ulster",
        "cavan": "Ulster",
        "donegal": "Ulster",
        "down": "Ulster",
        "fermanagh": "Ulster",
        "londonderry": "Ulster",
        "monaghan": "Ulster",
        "tyrone": "Ulster",
    }

    return county_to_province[county]


def clean_price_column(price: str):
    cleaned = price[2:-1].replace(',', '')
    return float(cleaned)


def clean_property_description(desc: str):
    if desc == 'Second-Hand Dwelling house /Apartment':
        return 'second-hand'
    return 'new'


def map_property_size(property_size_desc: str):
    if property_size_desc in ["less than 38 sq metres", "n?os l? n? 38 m?adar cearnach"]:
        return "small"
    if property_size_desc in ["greater than or equal to 38 sq metres and less than 125 sq metres",
                              "níos mó ná nó cothrom le 38 méadar cearnach agus níos lú ná 125 méadar cearnach"]:
        return "medium"
    if property_size_desc in ["greater than 125 sq metres", "greater than or equal to 125 sq metres"]:
        return "large"

    return np.nan


def convert_to_iso_date(date: str):
    for fmt in ("%d/%m/%y", "%d/%m/%Y"):
        try:
            return datetime.strptime(date, fmt)
        except ValueError:
            continue
    raise ValueError(f'Unknown date format: {date}')


def map_address(full_address: str):
    return full_address.replace(' ', '_').strip().lower()


def map_county(county: str):
    return 'Laoighis' if county == 'Laois' else county


df.columns = column_names

df['date_of_sale'] = df['date_of_sale'].map(convert_to_iso_date)
df['year'] = df['date_of_sale'].map(lambda date: date.year)
df['address'] = df['address'].map(map_address)
df['price'] = df['price'].map(clean_price_column)
df['province'] = df['county'].map(get_county_province_mapping)
df['county'] = df['county'].map(map_county)
df['not_full_market_price'] = df['not_full_market_price'].map(lambda s: s.lower())
df['vat_exclusive'] = df['vat_exclusive'].map(lambda s: True if s == 'Yes' else False)
df['property_size'] = df['propterty_size_description'].map(map_property_size)
df['property_description'] = df['property_description'].map(clean_property_description)

/var/folders/nh/yh9mg2mx39ng5t5l9bd8r2400000gn/T/ipykernel_81022/385170360.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./irish-property-sales.csv', encoding="latin1")


In [3]:
df.head()

,date_of_sale,address,county,eircode,price,not_full_market_price,vat_exclusive,property_description,propterty_size_description,year,province,property_size
0,2010-01-01,"5_braemor_drive,_churchtown,_co.dublin",Dublin,NaN,343000.0,no,False,second-hand,NaN,2010,Leinster,NaN
1,2010-01-03,"134_ashewood_walk,_summerhill_lane,_portlaoise",Laoighis,NaN,185000.0,no,True,new,greater than or equal to 38 sq metres and less...,2010,Leinster,medium
2,2010-01-04,"1_meadow_avenue,_dundrum,_dublin_14",Dublin,NaN,438500.0,no,False,second-hand,NaN,2010,Leinster,NaN
3,2010-01-04,"1_the_haven,_mornington",Meath,NaN,400000.0,no,False,second-hand,NaN,2010,Leinster,NaN
4,2010-01-04,"11_melville_heights,_kilkenny",Kilkenny,NaN,160000.0,no,False,second-hand,NaN,2010,Leinster,NaN


In [4]:
sales_volumn_per_county_per_year = df.groupby(["county", "year"])["price"].mean().reset_index()
result = []

for county, sub in sales_volumn_per_county_per_year.groupby("county"):
    avg_list = [
        {
            "year": int(row["year"]),
            "average": float(row["price"]),
        }
        for _, row in sub.iterrows()
    ]

    result.append({
        "county": str(county),
        "averagePrice": avg_list,
    })

with open("meanPriceCounties.json", "w") as f:
    json.dump(result, f, indent=2)


In [11]:
grouped = (
    df.groupby(['year', 'county', 'property_description'])
    .size()
    .reset_index(name='count')
    .pivot(index=['year', 'county'], columns='property_description', values='count')
    .fillna(0)
    .reset_index()
)

result = []
for _, row in grouped.iterrows():
    result.append({
        'year': int(row['year']),
        'county': row['county'],
        'new': int(row['new']),
        'secondHand': int(row['second-hand'])
    })

with open("numberOfPropertyTypesPerCountyYear.json", "w") as f:
    json.dump(result, f, indent=2)

In [14]:
df_year_county_price = df[['year', 'price','county', 'property_description']]
result = []
for year in df_year_county_price['year'].unique():
    df_year = df_year_county_price[df_year_county_price['year'] == year]

    for county in df_year['county'].unique():
        df_county = df_year[df_year['county'] == county]
        df_new = df_county[df_county['property_description'] == 'new']
        df_second_hand = df_county[df_county['property_description'] == 'second-hand']

        price_bins_new = pd.cut(df_new['price'], bins=bins, labels=labels, include_lowest=True)
        price_bins_second_hand = pd.cut(df_second_hand['price'], bins=bins, labels=labels, include_lowest=True)

        result.append({
            'year': int(year),
            'county': county,
            'priceBinsNew': price_bins_new.value_counts().sort_index().to_list(),
            'priceBinsSecondHand': price_bins_second_hand.value_counts().sort_index().to_list()
        })

with open("priceBinsPerCountyYear.json", "w") as f:
    json.dump(result, f, indent=2)


In [8]:
sales_volumn_per_county_per_year = df.groupby(["county", "year"]).count().reset_index()
print(sales_volumn_per_county_per_year.head())
result = []

for county, sub in sales_volumn_per_county_per_year.groupby("county"):
    avg_list = [
        {
            "year": int(row["year"]),
            "volumn": int(row["price"]),
        }
        for _, row in sub.iterrows()
    ]

    result.append({
        "county": str(county),
        "volumnPerYear": avg_list,
    })

with open("salesVolumnPerCounties.json", "w") as f:
    json.dump(result, f, indent=2)


   county  year  date_of_sale  address  eircode  price  not_full_market_price  \
0  Carlow  2010           231      231        0    231                    231   
1  Carlow  2011           183      183        0    183                    183   
2  Carlow  2012           266      266        0    266                    266   
3  Carlow  2013           375      375        0    375                    375   
4  Carlow  2014           438      438        2    438                    438   

   vat_exclusive  property_description  propterty_size_description  province  \
0            231                   231                          93       231   
1            183                   183                          48       183   
2            266                   266                          47       266   
3            375                   375                          66       375   
4            438                   438                          80       438   

   property_size  
0            